In [1]:
import os
import sys
import plotly.express as px
import logging
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, Javascript
sys.path.append("../../../..")
sys.path.append("../../../../scripts")
sys.path.append("../../../../scripts/summarize/calibration")

from notebooks.notebook_styling import bellevue_theme
from input_configuration import *
from h5toDF import *
from summary_functions import *
from dictionary import *
from utils import survey_year, get_subarea, get_data

logging.disable(logging.CRITICAL)

In [2]:
# data processing
taz_subarea = pd.read_csv(os.path.join(project_folder, districtfile))
taz_subarea['DistrictFlowName'] = taz_subarea['DistrictFlowID'].map(district_flow_name)
taz_subarea.rename(columns={'BKRCastTAZ': 'TAZ'}, inplace=True)
data_daysim = convert(os.path.join(project_folder, h5_results_file), 
                      os.path.join(project_folder, guidefile), 
                      os.path.join(project_folder, h5_results_name), stdout=False)
data_survey = convert(os.path.join(project_folder, h5_comparison_file),   # data_survey and data_fullsurvey are different in 2023, data_survey has a smaller number of records
                      os.path.join(project_folder, guidefile), 
                      os.path.join(project_folder, h5_comparison_name), stdout=False)
data_fullsurvey = convert(os.path.join(project_folder, h5_fullsurvey_file), 
                          os.path.join(project_folder, guidefile), 
                          os.path.join(project_folder, h5_fullsurvey_name), stdout=False)

data_survey['Trip_cloned']['mode'] = data_survey['Trip_cloned']['mode'].replace('TNC','Other')
data_fullsurvey['Trip']['mode'] = data_fullsurvey['Trip']['mode'].replace('TNC','Other')

# locate the ACS survey data
acs_data = os.path.join(project_folder, f'inputs/model/survey/ACS_2023.xlsx')
acs_data_bkr = os.path.join(project_folder, f'inputs/model/survey/ACS_2023_BKR.xlsx')

## Total Tours

In [3]:
def total_tours(data1, data2, tag='PSRC Region'):
    Tour_1_total = get_total(data1['Tour']['toexpfac'])
    Tour_2_total = get_total(data2['Tour_cloned']['toexpfac'])

    tpp  = pd.DataFrame(index = ['Tours'])
    tpp['DaysimOutputs'] = Tour_1_total
    tpp[f'{survey_year}Survey'] = Tour_2_total
    tpp = get_differences(tpp, 'DaysimOutputs', f'{survey_year}Survey', 2)
    # table
    display(tpp.style.format({
        'DaysimOutputs': '{:,.1f}',
        f'{survey_year}Survey': '{:,.1f}',
        f"Difference (DaysimOutputs - {survey_year}Survey)": '{:,.1f}',
        f"% Difference (DaysimOutputs - {survey_year}Survey)": '{:,.1f}%',}))

In [4]:
total_tours(data1=data_daysim, data2=data_survey, tag='PSRC Region')

,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Tours,"5,875,487.0","5,050,722.0","824,764.9",16.3%


In [5]:
import copy
_data_daysim = copy.deepcopy(data_daysim)
_data_survey = copy.deepcopy(data_survey)
_data_fullsurvey = copy.deepcopy(data_fullsurvey)
fname_tail, data_daysim_bkr, data_survey_bkr, data_fullsurvey_bkr = \
    get_data(data1=_data_daysim, data2=_data_survey, data3=_data_fullsurvey, taz_subarea=taz_subarea, if_region=False)
total_tours(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Tours,"462,173.0","332,686.9","129,486.1",38.9%


## Tour per Person

In [6]:
def tour_per_person(data1, data2, tag='PSRC Region'):
    Person_1_total = get_total(data1['Person']['psexpfac'])
    Person_2_total = get_total(data2['Person']['psexpfac'])
    Tour_1_total = get_total(data1['Tour']['toexpfac'])
    Tour_2_total = get_total(data2['Tour_cloned']['toexpfac'])

    ##Tours per person
    tpp1 = Tour_1_total / Person_1_total
    tpp2 = Tour_2_total / Person_2_total
    tpp  = pd.DataFrame(index = ['Tours'])
    tpp['DaysimOutputs'] = tpp1
    tpp[f'{survey_year}Survey'] = tpp2
    tpp = get_differences(tpp, 'DaysimOutputs', f'{survey_year}Survey', 2)
    # table
    display(tpp.style.format({
        'DaysimOutputs': '{:,.1f}',
        f'{survey_year}Survey': '{:,.1f}',
        f"Difference (DaysimOutputs - {survey_year}Survey)": '{:,.1f}',
        f"% Difference (DaysimOutputs - {survey_year}Survey)": '{:,.1f}%',}))

In [7]:
tour_per_person(data1=data_daysim, data2=data_survey, tag='PSRC Region')

,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Tours,1.4,1.3,0.0,2.2%


In [8]:
tour_per_person(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Tours,1.4,1.1,0.3,23.8%


## Tours per Person by Person Type

In [9]:
from collections import OrderedDict

def tour_by_pptyp(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Tours per Person by Purpose and Person Type/Number of Stops
    name1 = 'DaysimOutputs'
    name2 = f'{survey_year}Survey' 
    PersonsDay1 = pd.merge(data1['Person'][['hhno', 'pno', 'pptyp', 'psexpfac']], data1['PersonDay'][['hhno', 'pno', 'pdexpfac']], on= ['hhno', 'pno']).copy()
    PersonsDay2 = pd.merge(data2['Person'][['hhno', 'pno', 'pptyp', 'psexpfac']], data2['PersonDay_cloned'][['hhno', 'pno', 'pdexpfac']], on= ['hhno', 'pno']).copy()
    # calculate the percentage of each number of stops by purpose
    data1['Tour']['all_stops'] = data1['Tour']['tripsh1'] + data1['Tour']['tripsh2'] - 2  # minus the origin and destination
    data2['Tour_cloned']['all_stops'] = data2['Tour_cloned']['tripsh1'] + data2['Tour_cloned']['tripsh2'] - 2  # minus the origin and destination
    intermediate_stopsbypurp1 = data1['Tour'].groupby(['all_stops', 'pdpurp'])['toexpfac'].sum().reset_index()
    intermediate_stopsbypurp2 = data2['Tour_cloned'].groupby(['all_stops', 'pdpurp'])['toexpfac'].sum().reset_index()
    # retain only the first 5 stops
    intermediate_stopsbypurp1 = intermediate_stopsbypurp1[intermediate_stopsbypurp1['all_stops']<=5]
    intermediate_stopsbypurp2 = intermediate_stopsbypurp2[intermediate_stopsbypurp2['all_stops']<=5]

    for purpose in data1['Tour']['pdpurp'].value_counts().index:
        if purpose == 'Work':
            tc = 'wktours'
            sc = 'wkstops'
        elif purpose == 'Social':
            tc = 'sotours'
            sc = 'sostops'
        elif purpose == 'School':
            tc = 'sctours'
            sc = 'scstops'
        elif purpose == 'Escort':
            tc = 'estours'
            sc = 'esstops'
        elif purpose == 'Personal Business':
            tc = 'pbtours'
            sc = 'pbstops'
        elif purpose == 'Shop':
            tc = 'shtours'
            sc = 'shstops'
        elif purpose == 'Meal':
            tc = 'mltours'
            sc = 'mlstops'
        #Merge a column to PersonsDay for the current purpose
        PersonsDay1 = PersonsDay1.merge(data1['PersonDay'][['hhno', 'pno', tc]], on= ['hhno', 'pno'], how='left')
        PersonsDay2 = PersonsDay2.merge(data2['PersonDay_cloned'][['hhno', 'pno', tc]], on= ['hhno', 'pno'], how='left')
        toursPersPurp1 = weighted_average(PersonsDay1, tc, 'psexpfac', 'pptyp')
        toursPersPurp2 = weighted_average(PersonsDay2, tc, 'psexpfac', 'pptyp')
        #Delete added column to make future iterations faster
        PersonsDay1.drop(columns = [tc], inplace = True)
        PersonsDay2.drop(columns = [tc], inplace = True)
        items = OrderedDict(((name1, toursPersPurp1), (name2, toursPersPurp2)))
        toursPersPurp = pd.DataFrame.from_dict(items)
        toursPersPurp = get_differences(toursPersPurp, name1, name2, 2)
        toursPersPurp = recode_index(toursPersPurp, 'pptyp','Person Type')
        toursPersPurp = toursPersPurp.loc[ptype_cat.values(), :]
        # display table
        display(toursPersPurp.style.format({
            name1: '{:,.2f}',
            name2: '{:,.2f}',
            f'Difference ({name1} - {name2})': '{:,.2f}',
            f'% Difference ({name1} - {name2})': '{:,.2f}%',
        }))
        # bar plot
        fig = px.bar(
            toursPersPurp.reset_index(),
            x='Person Type',
            y=[name1, name2],
            barmode='group',
            title=f'{purpose} Tours by Person Type'
        )
        fig.update_layout(yaxis_title=f'{purpose} Tours per Person', xaxis_title='Person Type', xaxis=dict(showgrid=True), yaxis=dict(showgrid=True))
        fig.show()

        #Number of stops by purpose
        ps = pd.DataFrame()
        # fill missing 'all_stops' values with 0 for both intermediate_stopsbypurp1 and intermediate_stopsbypurp2
        imstpbypurp1 = intermediate_stopsbypurp1[intermediate_stopsbypurp1['pdpurp']==purpose].copy(deep=True)
        imstpbypurp2 = intermediate_stopsbypurp2[intermediate_stopsbypurp2['pdpurp']==purpose].copy(deep=True)
        # calculate percentage
        imstpbypurp1['percentage'] = imstpbypurp1['toexpfac'] / imstpbypurp1['toexpfac'].sum() * 100
        imstpbypurp2['percentage'] = imstpbypurp2['toexpfac'] / imstpbypurp2['toexpfac'].sum() * 100
        imstpbypurp1 = imstpbypurp1[imstpbypurp1['all_stops']<=5]
        imstpbypurp2 = imstpbypurp2[imstpbypurp2['all_stops']<=5]
        imstpbypurp1 = imstpbypurp1.set_index('all_stops').reindex(range(0, 6), fill_value=0).reset_index()
        imstpbypurp2 = imstpbypurp2.set_index('all_stops').reindex(range(0, 6), fill_value=0).reset_index()
        ps['% of Tours (' + name1 + ')'] = list(imstpbypurp1['percentage'])
        ps['% of Tours (' + name2 + ')'] = list(imstpbypurp2['percentage'])
        ps[purpose + ' Tours'] = range(0, 6)
        ps = ps.set_index(purpose + ' Tours')
        ps = get_differences( ps, '% of Tours (' + name1 + ')', '% of Tours (' + name2 + ')', 2)
        # table
        display(ps.style.format({
            f'% of Tours ({name1})': '{:,.1f}',
            f'% of Tours ({name2})': '{:,.1f}',
            f'Difference (% of Tours ({name1}) - % of Tours ({name2}))': '{:,.1f}',
            f'% Difference (% of Tours ({name1}) - % of Tours ({name2}))': '{:,.1f}%'
        }))

        # figure
        fig_stops = px.bar(
            ps.reset_index(),
            x=ps.index.name,
            y=[f'% of Tours ({name1})', f'% of Tours ({name2})'],
            barmode='group',
            title=f'Number of Stops per Tour by Purpose ({purpose}, {tag})'
        )
        fig_stops.update_layout(yaxis_title='Percentage of Tours', xaxis_title='Number of Stops', xaxis=dict(showgrid=True), yaxis=dict(showgrid=True))
        fig_stops.update_yaxes(ticksuffix='%')
        fig_stops.show()

In [10]:
tour_by_pptyp(data1=data_daysim, data2=data_survey, tag='PSRC Region')

,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Person Type,,,,
Full-Time Worker,0.66,0.66,0.00,0.56%
Part-Time Worker,0.61,0.52,0.09,18.10%
Non-Working Adult Age 65+,0.00,0.02,-0.02,-100.00%
Non-Working Adult Age <65,0.00,0.00,-0.00,-100.00%
University Student,0.39,0.21,0.18,84.49%
High School Student Age 16+,0.15,0.09,0.06,66.24%
Child Age 5-15,0.00,0.00,0.00,"5,601.30%"
Child Age 0-4,0.00,0.00,0.00,nan%


,% of Tours (DaysimOutputs),% of Tours (2023Survey),Difference (% of Tours (DaysimOutputs) - % of Tours (2023Survey)),% Difference (% of Tours (DaysimOutputs) - % of Tours (2023Survey))
Work Tours,,,,
0,35.1,53.2,-18.1,-34.0%
1,28.4,22.2,6.1,27.6%
2,19.5,14.4,5.1,35.8%
3,10.1,7.4,2.7,36.5%
4,4.8,2.2,2.6,116.3%
5,2.1,0.5,1.5,284.2%


,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Person Type,,,,
Full-Time Worker,0.31,0.23,0.07,31.69%
Part-Time Worker,0.35,0.31,0.05,14.94%
Non-Working Adult Age 65+,0.51,0.42,0.08,20.08%
Non-Working Adult Age <65,0.50,0.36,0.14,40.61%
University Student,0.21,0.32,-0.11,-34.26%
High School Student Age 16+,0.37,0.36,0.01,2.77%
Child Age 5-15,0.21,0.17,0.04,25.50%
Child Age 0-4,0.26,0.21,0.05,23.15%


,% of Tours (DaysimOutputs),% of Tours (2023Survey),Difference (% of Tours (DaysimOutputs) - % of Tours (2023Survey)),% Difference (% of Tours (DaysimOutputs) - % of Tours (2023Survey))
Social Tours,,,,
0,74.6,66.0,8.6,13.0%
1,18.0,19.2,-1.2,-6.3%
2,5.5,7.9,-2.4,-30.0%
3,1.4,3.5,-2.1,-59.6%
4,0.4,1.9,-1.5,-80.7%
5,0.1,1.5,-1.4,-94.8%


,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Person Type,,,,
Full-Time Worker,0.11,0.09,0.02,23.28%
Part-Time Worker,0.19,0.11,0.08,77.27%
Non-Working Adult Age 65+,0.30,0.25,0.05,18.91%
Non-Working Adult Age <65,0.43,0.27,0.16,58.18%
University Student,0.06,0.09,-0.03,-33.71%
High School Student Age 16+,0.06,0.06,-0.00,-1.81%
Child Age 5-15,0.11,0.03,0.08,255.65%
Child Age 0-4,0.27,0.09,0.18,189.48%


...

,% of Tours (DaysimOutputs),% of Tours (2023Survey),Difference (% of Tours (DaysimOutputs) - % of Tours (2023Survey)),% Difference (% of Tours (DaysimOutputs) - % of Tours (2023Survey))
School Tours,,,,
0,55.5,69.3,-13.8,-20.0%
1,23.8,24.9,-1.1,-4.4%
2,13.1,3.8,9.3,248.9%
3,5.2,1.3,4.0,306.9%
4,1.8,0.7,1.1,167.1%
5,0.6,0.1,0.5,699.1%


,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Person Type,,,,
Full-Time Worker,0.09,0.10,-0.01,-8.49%
Part-Time Worker,0.10,0.12,-0.01,-9.67%
Non-Working Adult Age 65+,0.09,0.09,0.00,4.27%
Non-Working Adult Age <65,0.15,0.07,0.08,108.15%
University Student,0.04,0.03,0.01,38.04%
High School Student Age 16+,0.10,0.06,0.04,74.17%
Child Age 5-15,0.01,0.00,0.01,987.86%
Child Age 0-4,0.09,0.05,0.04,86.09%


,% of Tours (DaysimOutputs),% of Tours (2023Survey),Difference (% of Tours (DaysimOutputs) - % of Tours (2023Survey)),% Difference (% of Tours (DaysimOutputs) - % of Tours (2023Survey))
Meal Tours,,,,
0,63.3,58.0,5.3,9.2%
1,23.4,22.8,0.6,2.4%
2,9.2,14.2,-5.0,-35.5%
3,3.0,3.1,-0.1,-2.4%
4,0.9,1.8,-1.0,-51.8%
5,0.2,0.1,0.2,280.5%


,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Person Type,,,,
Full-Time Worker,0.06,0.05,0.01,28.45%
Part-Time Worker,0.13,0.10,0.03,31.02%
Non-Working Adult Age 65+,0.25,0.15,0.10,71.35%
Non-Working Adult Age <65,0.22,0.14,0.08,55.54%
University Student,0.03,0.14,-0.11,-77.64%
High School Student Age 16+,0.03,0.03,0.00,3.11%
Child Age 5-15,0.02,0.02,-0.00,-8.44%
Child Age 0-4,0.06,0.06,0.00,7.39%


,% of Tours (DaysimOutputs),% of Tours (2023Survey),Difference (% of Tours (DaysimOutputs) - % of Tours (2023Survey)),% Difference (% of Tours (DaysimOutputs) - % of Tours (2023Survey))
Personal Business Tours,,,,
0,60.4,53.2,7.2,13.5%
1,25.7,28.9,-3.2,-11.0%
2,10.5,8.1,2.4,29.8%
3,2.6,4.8,-2.2,-46.2%
4,0.6,4.4,-3.8,-86.4%
5,0.1,0.5,-0.4,-77.3%


In [11]:
import copy
_data_daysim = copy.deepcopy(data_daysim)
_data_survey = copy.deepcopy(data_survey)
_data_fullsurvey = copy.deepcopy(data_fullsurvey)
fname_tail, data_daysim_bkr, data_survey_bkr, data_fullsurvey_bkr = \
    get_data(data1=_data_daysim, data2=_data_survey, data3=_data_fullsurvey, taz_subarea=taz_subarea, if_region=False)
tour_by_pptyp(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Person Type,,,,
Full-Time Worker,0.38,0.22,0.17,76.09%
Part-Time Worker,0.43,0.36,0.07,19.20%
Non-Working Adult Age 65+,0.57,0.25,0.32,127.19%
Non-Working Adult Age <65,0.57,0.71,-0.14,-19.43%
University Student,0.26,0.44,-0.18,-40.85%
High School Student Age 16+,0.44,0.58,-0.13,-22.98%
Child Age 5-15,0.25,0.20,0.04,21.44%
Child Age 0-4,0.31,0.09,0.22,252.09%


,% of Tours (DaysimOutputs),% of Tours (2023Survey),Difference (% of Tours (DaysimOutputs) - % of Tours (2023Survey)),% Difference (% of Tours (DaysimOutputs) - % of Tours (2023Survey))
Social Tours,,,,
0,74.7,59.7,15.0,25.1%
1,17.9,28.4,-10.4,-36.8%
2,5.5,4.7,0.8,16.0%
3,1.4,1.1,0.3,29.6%
4,0.4,3.1,-2.7,-87.4%
5,0.1,3.0,-2.9,-97.5%


,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Person Type,,,,
Full-Time Worker,0.62,0.63,-0.01,-1.43%
Part-Time Worker,0.55,0.21,0.34,160.60%
Non-Working Adult Age 65+,0.00,0.00,-0.00,-100.00%
Non-Working Adult Age <65,0.00,0.00,0.00,nan%
University Student,0.33,0.04,0.28,671.85%
High School Student Age 16+,0.15,0.08,0.06,74.66%
Child Age 5-15,0.00,0.00,0.00,nan%
Child Age 0-4,0.00,0.00,0.00,nan%


,% of Tours (DaysimOutputs),% of Tours (2023Survey),Difference (% of Tours (DaysimOutputs) - % of Tours (2023Survey)),% Difference (% of Tours (DaysimOutputs) - % of Tours (2023Survey))
Work Tours,,,,
0,37.5,46.9,-9.3,-19.9%
1,27.4,27.8,-0.4,-1.5%
2,18.8,21.1,-2.3,-11.1%
3,9.7,2.6,7.0,264.4%
4,4.7,1.6,3.0,188.9%
5,2.0,0.0,2.0,nan%


,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Person Type,,,,
Full-Time Worker,0.00,0.00,0.00,nan%
Part-Time Worker,0.00,0.00,-0.00,-100.00%
Non-Working Adult Age 65+,0.00,0.00,0.00,nan%
Non-Working Adult Age <65,0.00,0.00,-0.00,-100.00%
University Student,0.63,0.26,0.37,141.01%
High School Student Age 16+,0.74,0.25,0.49,198.45%
Child Age 5-15,0.74,0.83,-0.10,-11.72%
Child Age 0-4,0.00,0.27,-0.27,-100.00%


...

,% of Tours (DaysimOutputs),% of Tours (2023Survey),Difference (% of Tours (DaysimOutputs) - % of Tours (2023Survey)),% Difference (% of Tours (DaysimOutputs) - % of Tours (2023Survey))
Shop Tours,,,,
0,52.3,67.2,-14.9,-22.2%
1,25.1,20.8,4.3,20.6%
2,14.8,8.3,6.4,76.8%
3,5.3,1.7,3.6,211.6%
4,1.9,1.6,0.4,23.7%
5,0.6,0.4,0.2,50.1%


,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Person Type,,,,
Full-Time Worker,0.12,0.15,-0.03,-19.35%
Part-Time Worker,0.14,0.34,-0.20,-58.27%
Non-Working Adult Age 65+,0.12,0.03,0.08,254.49%
Non-Working Adult Age <65,0.18,0.01,0.17,"1,845.36%"
University Student,0.06,0.04,0.03,77.06%
High School Student Age 16+,0.13,0.05,0.08,143.43%
Child Age 5-15,0.02,0.00,0.02,"5,327.51%"
Child Age 0-4,0.11,0.00,0.11,nan%


,% of Tours (DaysimOutputs),% of Tours (2023Survey),Difference (% of Tours (DaysimOutputs) - % of Tours (2023Survey)),% Difference (% of Tours (DaysimOutputs) - % of Tours (2023Survey))
Meal Tours,,,,
0,63.2,86.4,-23.2,-26.8%
1,23.8,8.7,15.1,173.3%
2,9.1,4.0,5.0,125.6%
3,2.8,0.9,1.9,215.3%
4,0.9,0.0,0.9,nan%
5,0.2,0.0,0.2,nan%


,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Person Type,,,,
Full-Time Worker,0.07,0.03,0.04,142.45%
Part-Time Worker,0.14,0.06,0.09,157.92%
Non-Working Adult Age 65+,0.24,0.17,0.07,42.10%
Non-Working Adult Age <65,0.20,0.06,0.14,243.76%
University Student,0.03,0.01,0.02,273.00%
High School Student Age 16+,0.03,0.05,-0.03,-49.72%
Child Age 5-15,0.02,0.00,0.02,"1,388.61%"
Child Age 0-4,0.06,0.00,0.06,"1,807.87%"


,% of Tours (DaysimOutputs),% of Tours (2023Survey),Difference (% of Tours (DaysimOutputs) - % of Tours (2023Survey)),% Difference (% of Tours (DaysimOutputs) - % of Tours (2023Survey))
Personal Business Tours,,,,
0,60.2,63.2,-3.0,-4.7%
1,25.8,32.3,-6.5,-20.1%
2,10.6,2.4,8.1,334.2%
3,2.7,2.1,0.6,30.8%
4,0.6,0.0,0.6,nan%
5,0.1,0.0,0.1,nan%


## Tours per Person by Purpose

In [12]:
def tours_per_ps_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Tours per Person by Purpose
    Person_1_total = get_total(data1['Person']['psexpfac'])
    Person_2_total = get_total(data2['Person']['psexpfac'])
    tpbp1 = data1['Tour'][['pdpurp','toexpfac']].groupby('pdpurp').sum()['toexpfac'] / Person_1_total
    tpbp2 = data2['Tour_cloned'][['pdpurp','toexpfac']].groupby('pdpurp').sum()['toexpfac'] / Person_2_total
    tpbp = pd.DataFrame()
    tpbp['Tours per Person (DaysimOutputs)'] = tpbp1
    tpbp[f'Tours per Person ({survey_year}Survey)'] = tpbp2
    tpbp = get_differences(tpbp, 'Tours per Person (DaysimOutputs)', 'Tours per Person ({survey_year}Survey)', 2)
    tpbp = recode_index(tpbp, 'pdpurp', 'Tour Purpose')
    tpbp = tpbp.loc[pdpurp_cat.values()]
    # table
    display(tpbp.style.format({
        'Tours per Person (DaysimOutputs)': '{:,.1f}',
        f'Tours per Person ({survey_year}Survey)': '{:,.1f}',
        f"Difference (Tours per Person (DaysimOutputs) - Tours per Person ({survey_year}Survey))": '{:,.1f}',
        f"% Difference (Tours per Person (DaysimOutputs) - Tours per Person ({survey_year}Survey))": '{:,.1f}%',
    }))
    # plot
    fig = px.bar(
        tpbp.reset_index(),
        x='Tour Purpose',
        y=['Tours per Person (DaysimOutputs)', f'Tours per Person ({survey_year}Survey)'],
        barmode='group',
        title=f'Tours per Person by Purpose ({tag})'
    )
    fig.update_layout(yaxis_title='Tours per Person', xaxis_title='Tour Purpose', xaxis=dict(showgrid=True), yaxis=dict(showgrid=True))
    fig.show()    

In [13]:
tours_per_ps_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region')

,Tours per Person (DaysimOutputs),Tours per Person (2023Survey),Difference (Tours per Person (DaysimOutputs) - Tours per Person (2023Survey)),% Difference (Tours per Person (DaysimOutputs) - Tours per Person (2023Survey))
Tour Purpose,,,,
Work,0.4,0.4,0.0,2.3%
School,0.1,0.1,0.0,2.8%
Escort,0.1,0.1,0.0,4.7%
Personal Business,0.1,0.1,0.0,4.6%
Shop,0.2,0.2,0.0,2.1%
Meal,0.1,0.1,0.0,4.4%
Social,0.3,0.3,-0.0,-0.5%


In [14]:
tours_per_ps_by_purp(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

,Tours per Person (DaysimOutputs),Tours per Person (2023Survey),Difference (Tours per Person (DaysimOutputs) - Tours per Person (2023Survey)),% Difference (Tours per Person (DaysimOutputs) - Tours per Person (2023Survey))
Tour Purpose,,,,
Work,0.3,0.3,0.0,11.7%
School,0.2,0.1,0.0,10.9%
Escort,0.1,0.1,0.0,15.9%
Personal Business,0.1,0.1,0.0,87.9%
Shop,0.1,0.1,0.1,61.2%
Meal,0.1,0.1,0.1,60.0%
Social,0.4,0.3,0.1,16.2%


## Tours per Person by Mode

In [15]:
from collections import OrderedDict

def tour_by_mode(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Tours per Person by Purpose and Person Type/Number of Stops
    name1 = 'DaysimOutputs'
    name2 = f'{survey_year}Survey' 
    PersonsDay1 = pd.merge(data1['Tour'][['hhno', 'pno', 'tmodetp']], data1['PersonDay'][['hhno', 'pno', 'pdexpfac']], on= ['hhno', 'pno']).copy()
    PersonsDay2 = pd.merge(data2['Tour_cloned'][['hhno', 'pno', 'tmodetp']], data2['PersonDay_cloned'][['hhno', 'pno', 'pdexpfac']], on= ['hhno', 'pno']).copy()
    # calculate the percentage of each number of stops by purpose
    data1['Tour']['all_stops'] = data1['Tour']['tripsh1'] + data1['Tour']['tripsh2'] - 2  # minus the origin and destination
    data2['Tour_cloned']['all_stops'] = data2['Tour_cloned']['tripsh1'] + data2['Tour_cloned']['tripsh2'] - 2  # minus the origin and destination
    intermediate_stopsbypurp1 = data1['Tour'].groupby(['all_stops', 'pdpurp'])['toexpfac'].sum().reset_index()
    intermediate_stopsbypurp2 = data2['Tour_cloned'].groupby(['all_stops', 'pdpurp'])['toexpfac'].sum().reset_index()
    # retain only the first 5 stops
    intermediate_stopsbypurp1 = intermediate_stopsbypurp1[intermediate_stopsbypurp1['all_stops']<=5]
    intermediate_stopsbypurp2 = intermediate_stopsbypurp2[intermediate_stopsbypurp2['all_stops']<=5]

    for purpose in data1['Tour']['pdpurp'].value_counts().index:
        if purpose == 'Work':
            tc = 'wktours'
            sc = 'wkstops'
        elif purpose == 'Social':
            tc = 'sotours'
            sc = 'sostops'
        elif purpose == 'School':
            tc = 'sctours'
            sc = 'scstops'
        elif purpose == 'Escort':
            tc = 'estours'
            sc = 'esstops'
        elif purpose == 'Personal Business':
            tc = 'pbtours'
            sc = 'pbstops'
        elif purpose == 'Shop':
            tc = 'shtours'
            sc = 'shstops'
        elif purpose == 'Meal':
            tc = 'mltours'
            sc = 'mlstops'
        #Merge a column to PersonsDay for the current purpose
        PersonsDay1 = PersonsDay1.merge(data1['PersonDay'][['hhno', 'pno', tc]], on= ['hhno', 'pno'], how='left')
        PersonsDay2 = PersonsDay2.merge(data2['PersonDay_cloned'][['hhno', 'pno', tc]], on= ['hhno', 'pno'], how='left')
        toursPersPurp1 = weighted_average(PersonsDay1, tc, 'pdexpfac', 'tmodetp')
        toursPersPurp2 = weighted_average(PersonsDay2, tc, 'pdexpfac', 'tmodetp')
        #Delete added column to make future iterations faster
        PersonsDay1.drop(columns = [tc], inplace = True)
        PersonsDay2.drop(columns = [tc], inplace = True)
        items = OrderedDict(((name1, toursPersPurp1), (name2, toursPersPurp2)))
        toursPersPurp = pd.DataFrame.from_dict(items)
        toursPersPurp = get_differences(toursPersPurp, name1, name2, 2)
        toursPersPurp = recode_index(toursPersPurp, 'tmodetp','Mode')
        # Retain only rows where the index (mode) is not 'Other'
        toursPersPurp = toursPersPurp[toursPersPurp.index != 'Other']
        toursPersPurp = toursPersPurp.loc[mode_cat.values(), :]
        # display table
        display(toursPersPurp.style.format({
            name1: '{:,.1f}',
            name2: '{:,.1f}',
            f'Difference ({name1} - {name2})': '{:,.1f}',
            f'% Difference ({name1} - {name2})': '{:,.1f}%',
        }))
        # bar plot
        fig = px.bar(
            toursPersPurp.reset_index(),
            x='Mode',
            y=[name1, name2],
            barmode='group',
            title=f'{purpose} Tours by Mode'
        )
        fig.update_layout(yaxis_title=f'{purpose} Tours per Person', xaxis_title='Mode', xaxis=dict(showgrid=True), yaxis=dict(showgrid=True))
        fig.show()

In [16]:
tour_by_mode(data1=data_daysim, data2=data_survey, tag='PSRC Region')

,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Mode,,,,
Walk,0.4,0.5,-0.0,-8.4%
Bike,0.2,0.5,-0.3,-61.4%
SOV,0.7,0.6,0.0,1.7%
HOV2,0.3,0.3,-0.0,-8.7%
HOV3+,0.3,0.2,0.1,26.0%
Transit Walk Access,0.5,0.3,0.2,49.9%
Transit Auto Access,1.1,0.6,0.5,80.6%
School Bus,0.0,0.0,0.0,nan%


,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Mode,,,,
Walk,0.7,0.9,-0.2,-19.7%
Bike,0.7,0.5,0.1,25.2%
SOV,0.5,0.4,0.1,22.7%
HOV2,0.5,0.5,0.0,6.1%
HOV3+,0.5,0.5,-0.0,-3.5%
Transit Walk Access,0.4,0.3,0.1,19.5%
Transit Auto Access,0.2,0.4,-0.1,-34.8%
School Bus,0.2,0.1,0.1,36.5%


,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Mode,,,,
Walk,0.3,0.3,0.0,1.5%
Bike,0.4,0.1,0.4,580.1%
SOV,0.3,0.2,0.0,4.8%
HOV2,0.4,0.2,0.1,55.2%
HOV3+,0.4,0.2,0.2,132.7%
Transit Walk Access,0.2,0.3,-0.1,-34.3%
Transit Auto Access,0.1,0.0,0.0,77.7%
School Bus,0.0,0.0,0.0,"1,015.7%"


,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Mode,,,,
Walk,0.2,0.2,-0.0,-13.0%
Bike,0.2,0.2,0.0,12.1%
SOV,0.1,0.1,0.0,4.5%
HOV2,0.4,0.4,-0.0,-4.1%
HOV3+,0.5,0.5,0.0,6.2%
Transit Walk Access,0.1,0.0,0.0,160.7%
Transit Auto Access,0.0,0.0,0.0,nan%
School Bus,0.1,0.1,-0.0,-24.8%


,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Mode,,,,
Walk,0.1,0.1,0.0,33.2%
Bike,0.3,0.2,0.1,48.0%
SOV,0.1,0.1,0.0,74.8%
HOV2,0.1,0.1,-0.0,-23.0%
HOV3+,0.2,0.3,-0.1,-37.7%
Transit Walk Access,0.3,0.1,0.2,133.1%
Transit Auto Access,0.0,0.0,0.0,nan%
School Bus,1.0,1.0,0.0,4.5%


,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Mode,,,,
Walk,0.2,0.2,0.0,0.0%
Bike,0.1,0.1,-0.0,-27.5%
SOV,0.1,0.1,0.0,7.2%
HOV2,0.2,0.2,-0.0,-21.1%
HOV3+,0.1,0.2,-0.0,-27.8%
Transit Walk Access,0.1,0.0,0.1,190.2%
Transit Auto Access,0.1,0.0,0.0,80.7%
School Bus,0.0,0.0,0.0,"2,028.3%"


,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Mode,,,,
Walk,0.2,0.1,0.1,92.1%
Bike,0.1,0.2,-0.0,-15.9%
SOV,0.2,0.1,0.0,18.0%
HOV2,0.1,0.1,0.0,4.8%
HOV3+,0.1,0.1,0.0,28.1%
Transit Walk Access,0.2,0.2,-0.0,-6.0%
Transit Auto Access,0.0,0.0,0.0,16.1%
School Bus,0.0,0.1,-0.1,-76.5%


In [17]:
tour_by_mode(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Mode,,,,
Walk,0.8,0.9,-0.1,-15.6%
Bike,0.9,0.5,0.3,66.7%
SOV,0.6,0.5,0.2,40.1%
HOV2,0.6,0.5,0.1,23.1%
HOV3+,0.6,0.7,-0.2,-25.9%
Transit Walk Access,0.6,0.4,0.2,51.4%
Transit Auto Access,0.3,0.7,-0.4,-51.9%
School Bus,0.2,0.3,-0.0,-10.0%


,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Mode,,,,
Walk,0.5,0.4,0.0,7.2%
Bike,0.2,0.2,0.1,29.1%
SOV,0.6,0.7,-0.1,-15.1%
HOV2,0.3,0.2,0.1,18.7%
HOV3+,0.3,0.2,0.1,21.2%
Transit Walk Access,0.4,0.3,0.2,58.9%
Transit Auto Access,1.1,0.3,0.8,281.3%
School Bus,0.0,0.0,0.0,nan%


,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Mode,,,,
Walk,0.1,0.1,-0.0,-2.3%
Bike,0.2,0.5,-0.3,-56.6%
SOV,0.1,0.0,0.1,429.1%
HOV2,0.1,0.2,-0.0,-27.9%
HOV3+,0.2,0.4,-0.1,-41.3%
Transit Walk Access,0.2,0.4,-0.2,-41.2%
Transit Auto Access,0.0,0.0,0.0,nan%
School Bus,1.0,1.1,-0.1,-9.7%


,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Mode,,,,
Walk,0.2,0.3,-0.1,-44.3%
Bike,0.3,0.6,-0.3,-49.5%
SOV,0.1,0.1,0.0,51.4%
HOV2,0.4,0.5,-0.1,-11.4%
HOV3+,0.5,0.5,0.1,10.5%
Transit Walk Access,0.1,0.0,0.1,"3,756.7%"
Transit Auto Access,0.0,0.0,0.0,nan%
School Bus,0.1,0.0,0.1,nan%


,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Mode,,,,
Walk,0.2,0.1,0.1,85.8%
Bike,0.2,0.0,0.2,nan%
SOV,0.2,0.2,0.1,33.0%
HOV2,0.2,0.1,0.1,67.2%
HOV3+,0.2,0.1,0.0,17.5%
Transit Walk Access,0.2,0.1,0.1,108.7%
Transit Auto Access,0.1,0.0,0.1,nan%
School Bus,0.0,0.0,0.0,645.3%


,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Mode,,,,
Walk,0.2,0.2,-0.0,-13.2%
Bike,0.1,0.0,0.1,"3,619.1%"
SOV,0.2,0.1,0.1,47.9%
HOV2,0.2,0.1,0.1,57.0%
HOV3+,0.2,0.1,0.1,69.3%
Transit Walk Access,0.2,0.0,0.1,362.9%
Transit Auto Access,0.1,0.0,0.1,nan%
School Bus,0.0,0.0,0.0,nan%


,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Mode,,,,
Walk,0.1,0.0,0.1,"1,072.9%"
Bike,0.1,0.0,0.1,nan%
SOV,0.2,0.2,0.0,10.8%
HOV2,0.1,0.1,0.1,51.0%
HOV3+,0.1,0.0,0.1,166.0%
Transit Walk Access,0.2,0.1,0.1,262.5%
Transit Auto Access,0.1,0.0,0.1,nan%
School Bus,0.0,0.0,0.0,nan%


## Tour Share by Purpose

In [18]:
def pc_tour_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Percent of Tours by Purpose
    tour_1_total = get_total(data_daysim['Tour']['toexpfac'])
    tour_2_total = get_total(data_survey['Tour_cloned']['toexpfac'])
    ptbp1 = 100 * data1['Tour'][['pdpurp','toexpfac']].groupby('pdpurp').sum()['toexpfac'] / tour_1_total
    ptbp2 = 100 * data2['Tour_cloned'][['pdpurp','toexpfac']].groupby('pdpurp').sum()['toexpfac'] / tour_2_total
    ptbp = pd.DataFrame()
    ptbp['Percent of Tours (DaysimOutputs)'] = ptbp1
    ptbp[f'Percent of Tours ({survey_year}Survey)'] = ptbp2
    ptbp = get_differences(ptbp,'Percent of Tours (DaysimOutputs)', f'Percent of Tours ({survey_year}Survey)', 2)
    ptbp = recode_index(ptbp, 'pdpurp', 'Tour Purpose')
    ptbp = ptbp.loc[pdpurp_cat.values()]
    # table
    display(ptbp.style.format({
        'Percent of Tours (DaysimOutputs)': '{:,.1f}%',
        f'Percent of Tours ({survey_year}Survey)': '{:,.1f}%',
        f"Difference (Percent of Tours (DaysimOutputs) - Percent of Tours ({survey_year}Survey))": '{:,.1f}%',
        f"% Difference (Percent of Tours (DaysimOutputs) - Percent of Tours ({survey_year}Survey))": '{:,.1f}%',
    }))
    # plot
    fig = px.bar(
        ptbp.reset_index(),
        x='Tour Purpose',
        y=['Percent of Tours (DaysimOutputs)', f'Percent of Tours ({survey_year}Survey)'],
        barmode='group',
        title=f'Percent of Tours by Purpose ({tag})'
    )
    fig.update_layout(yaxis_title='Percent of Tours', xaxis_title='Tour Purpose', xaxis=dict(showgrid=True), yaxis=dict(showgrid=True))
    fig.update_yaxes(ticksuffix='%')
    fig.show()

In [19]:
pc_tour_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region')

,Percent of Tours (DaysimOutputs),Percent of Tours (2023Survey),Difference (Percent of Tours (DaysimOutputs) - Percent of Tours (2023Survey)),% Difference (Percent of Tours (DaysimOutputs) - Percent of Tours (2023Survey))
Tour Purpose,,,,
Work,27.6%,27.5%,0.0%,0.1%
School,9.8%,9.7%,0.1%,0.7%
Escort,11.1%,10.8%,0.3%,2.5%
Personal Business,7.0%,6.8%,0.2%,2.4%
Shop,12.9%,12.9%,-0.0%,-0.1%
Meal,7.1%,7.0%,0.1%,2.1%
Social,24.6%,25.3%,-0.7%,-2.7%


In [20]:
pc_tour_by_purp(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

,Percent of Tours (DaysimOutputs),Percent of Tours (2023Survey),Difference (Percent of Tours (DaysimOutputs) - Percent of Tours (2023Survey)),% Difference (Percent of Tours (DaysimOutputs) - Percent of Tours (2023Survey))
Tour Purpose,,,,
Work,1.9%,1.8%,0.1%,7.7%
School,0.9%,0.8%,0.1%,7.0%
Escort,0.8%,0.8%,0.1%,11.8%
Personal Business,0.5%,0.3%,0.2%,81.2%
Shop,0.8%,0.5%,0.3%,55.5%
Meal,0.7%,0.4%,0.2%,54.2%
Social,2.2%,2.0%,0.2%,12.1%


## Tour Share by Mode

In [21]:
def pc_tour_by_mode(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Percent of Tours by Mode
    tour_1_total = get_total(data_daysim['Tour']['toexpfac'])
    tour_2_total = get_total(data_survey['Tour_cloned']['toexpfac'])
    ptbp1 = 100 * data1['Tour'][['tmodetp','toexpfac']].groupby('tmodetp').sum()['toexpfac'] / tour_1_total
    ptbp2 = 100 * data2['Tour_cloned'][['tmodetp','toexpfac']].groupby('tmodetp').sum()['toexpfac'] / tour_2_total
    ptbp = pd.DataFrame()
    ptbp['Percent of Tours (DaysimOutputs)'] = ptbp1
    ptbp[f'Percent of Tours ({survey_year}Survey)'] = ptbp2
    ptbp = get_differences(ptbp,'Percent of Tours (DaysimOutputs)', f'Percent of Tours ({survey_year}Survey)', 2)
    ptbp = recode_index(ptbp, 'tmodetp', 'Tour Mode')
    ptbp = ptbp.loc[mode_cat.values()]
    # table
    display(ptbp.style.format({
        'Percent of Tours (DaysimOutputs)': '{:,.1f}%',
        f'Percent of Tours ({survey_year}Survey)': '{:,.1f}%',
        f"Difference (Percent of Tours (DaysimOutputs) - Percent of Tours ({survey_year}Survey))": '{:,.1f}%',
        f"% Difference (Percent of Tours (DaysimOutputs) - Percent of Tours ({survey_year}Survey))": '{:,.1f}%',
    }))
    # plot
    fig = px.bar(
        ptbp.reset_index(),
        x='Tour Mode',
        y=['Percent of Tours (DaysimOutputs)', f'Percent of Tours ({survey_year}Survey)'],
        barmode='group',
        title=f'Tour Share by Mode ({tag})'
    )
    fig.update_layout(yaxis_title='Tour Share', xaxis_title='Tour Mode', xaxis=dict(showgrid=True), yaxis=dict(showgrid=True))
    fig.update_yaxes(ticksuffix='%')
    fig.show()

In [22]:
pc_tour_by_mode(data1=data_daysim, data2=data_survey, tag='PSRC Region')

,Percent of Tours (DaysimOutputs),Percent of Tours (2023Survey),Difference (Percent of Tours (DaysimOutputs) - Percent of Tours (2023Survey)),% Difference (Percent of Tours (DaysimOutputs) - Percent of Tours (2023Survey))
Tour Mode,,,,
Walk,10.8%,10.9%,-0.2%,-1.4%
Bike,0.6%,1.2%,-0.7%,-55.4%
SOV,38.6%,37.3%,1.3%,3.6%
HOV2,22.4%,22.8%,-0.4%,-1.8%
HOV3+,21.6%,19.5%,2.1%,10.8%
Transit Walk Access,3.5%,4.3%,-0.8%,-19.4%
Transit Auto Access,0.2%,0.4%,-0.2%,-49.6%
School Bus,2.4%,2.5%,-0.1%,-3.0%


In [23]:
pc_tour_by_mode(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

,Percent of Tours (DaysimOutputs),Percent of Tours (2023Survey),Difference (Percent of Tours (DaysimOutputs) - Percent of Tours (2023Survey)),% Difference (Percent of Tours (DaysimOutputs) - Percent of Tours (2023Survey))
Tour Mode,,,,
Walk,0.6%,0.8%,-0.2%,-28.2%
Bike,0.0%,0.1%,-0.0%,-52.2%
SOV,3.0%,2.2%,0.8%,38.5%
HOV2,1.9%,1.6%,0.3%,17.5%
HOV3+,1.6%,1.1%,0.5%,48.1%
Transit Walk Access,0.5%,0.3%,0.2%,64.6%
Transit Auto Access,0.0%,0.2%,-0.1%,-87.4%
School Bus,0.2%,0.2%,0.0%,2.2%


## Tour Distance by Purpose

In [24]:
def tours_distance_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Average distance by tour purpose
    name1 = 'DaysimOutputs'
    name2 = f'{survey_year}Survey' 
    ##Filter out unreasonable trip/tour lengths
    ##survey data does not include drive to transit trips, remove- how can we do this without referencing max internal zone?    
    tu_hh_1 = data1['Tour'].merge(data1['Household'][['hhno', 'hhtaz']], on='hhno')
    tu_hh_2 = data2['Tour_cloned'].merge(data2['Household'][['hhno', 'hhtaz']], on='hhno')
    tp_hh_1 = data1['Trip'].merge(data1['Household'][['hhno', 'hhtaz']], on='hhno')
    tp_hh_2 = data2['Trip_cloned'].merge(data2['Household'][['hhno', 'hhtaz']], on='hhno')

    tour_ok_1 = tu_hh_1.\
        query('tautodist>0 and tautodist<200')[['hhno', 'pno', 'tour', 'day', 'tautodist', 'toexpfac', 'pdpurp', 'tmodetp', 'tdtaz']].copy(deep=True)
    tour_ok_2 = tu_hh_2.\
        query('tautodist>0 and tautodist<200')[['hhno', 'pno', 'tour', 'day', 'tautodist', 'toexpfac', 'pdpurp', 'tmodetp', 'tdtaz']].copy(deep=True) 
    trip_ok_1 = tp_hh_1.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'tour', 'day', 'travdist', 'trexpfac', 'dpurp', 'mode', 'dtaz']].copy(deep=True)
    trip_ok_2 = tp_hh_2.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'tour', 'day', 'travdist', 'trexpfac', 'dpurp', 'mode', 'dtaz']] .copy(deep=True)
    
    #Merge tour and trip files
    tourtrip1 = pd.merge(tour_ok_1[['hhno', 'pno', 'tour', 'day', 'tautodist', 'toexpfac', 'pdpurp', 'tmodetp']],
                       trip_ok_1[['hhno', 'pno', 'tour', 'day', 'trexpfac']],
                       on = ['hhno', 'pno', 'tour', 'day'])
    tourtrip2 = pd.merge(tour_ok_2[['hhno', 'pno', 'tour', 'day', 'tautodist', 'toexpfac', 'pdpurp', 'tmodetp']],
                       trip_ok_2[['hhno', 'pno', 'tour', 'day', 'trexpfac']],
                       on = ['hhno', 'pno', 'tour', 'day'])

    #Compute weighted average of trip length grouped by purpose
    triptotal1 = weighted_average(tourtrip1[['tautodist', 'toexpfac', 'pdpurp']], 'tautodist', 'toexpfac', 'pdpurp')
    triptotal2 = weighted_average(tourtrip2[['tautodist', 'toexpfac', 'pdpurp']], 'tautodist', 'toexpfac', 'pdpurp')

    #Create data frame
    atl1 = pd.DataFrame.from_dict({'Average Tour Length (' + name1 + ')' : triptotal1})
    atl2 = pd.DataFrame.from_dict({'Average Tour Length (' + name2 + ')': triptotal2})
    atl = pd.merge(atl1, atl2, 'outer', left_index = True, right_index = True)
    atl = get_differences(atl, 'Average Tour Length (' + name1 + ')', 'Average Tour Length (' + name2 + ')', 2)
    atl = recode_index(atl, 'pdpurp', 'Tour Purpose')  
    atl = atl.loc[pdpurp_cat.values()]
    # display table
    display(atl.style.format({
        f'Average Tour Length ({name1})': '{:,.1f}',
        f'Average Tour Length ({name2})': '{:,.1f}',
        f'Difference (Average Tour Length ({name1}) - Average Tour Length ({name2}))': '{:,.1f}',
        f'% Difference (Average Tour Length ({name1}) - Average Tour Length ({name2}))': '{:,.1f}%'
    }))
    # bar plot
    fig = px.bar(
        atl.reset_index(),
        x='Tour Purpose',
        y=[f'Average Tour Length ({name1})', f'Average Tour Length ({name2})'],
        barmode='group',
        title=f'Average Tour Distance by Purpose ({tag})'
    )
    fig.update_layout(yaxis_title='Average Tour Distance', xaxis_title='Tour Purpose', xaxis=dict(showgrid=True), yaxis=dict(showgrid=True))
    fig.show()

In [25]:
tours_distance_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region')

,Average Tour Length (DaysimOutputs),Average Tour Length (2023Survey),Difference (Average Tour Length (DaysimOutputs) - Average Tour Length (2023Survey)),% Difference (Average Tour Length (DaysimOutputs) - Average Tour Length (2023Survey))
Tour Purpose,,,,
Work,11.6,11.8,-0.2,-2.0%
School,4.8,4.6,0.2,5.3%
Escort,7.4,7.1,0.3,4.7%
Personal Business,6.7,8.0,-1.2,-15.7%
Shop,4.8,4.7,0.1,2.0%
Meal,3.6,3.8,-0.1,-3.8%
Social,5.8,6.9,-1.1,-16.3%


In [26]:
tours_distance_by_purp(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

,Average Tour Length (DaysimOutputs),Average Tour Length (2023Survey),Difference (Average Tour Length (DaysimOutputs) - Average Tour Length (2023Survey)),% Difference (Average Tour Length (DaysimOutputs) - Average Tour Length (2023Survey))
Tour Purpose,,,,
Work,9.2,7.0,2.2,31.1%
School,3.9,4.3,-0.3,-8.2%
Escort,6.6,11.2,-4.6,-41.3%
Personal Business,7.1,5.7,1.4,25.1%
Shop,2.7,4.0,-1.3,-32.0%
Meal,2.9,5.3,-2.4,-46.0%
Social,4.9,8.0,-3.1,-39.0%


## Tour Distance by Mode

In [27]:
def tour_distance_by_mode(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    #Average Distance by Tour Mode
    name1 = 'DaysimOutputs'
    name2 = f'{survey_year}Survey' 
    ##Filter out unreasonable trip/tour lengths
    ##survey data does not include drive to transit trips, remove- how can we do this without referencing max internal zone?    
    tu_hh_1 = data1['Tour'].merge(data1['Household'][['hhno', 'hhtaz']], on='hhno')
    tu_hh_2 = data2['Tour_cloned'].merge(data2['Household'][['hhno', 'hhtaz']], on='hhno')
    tp_hh_1 = data1['Trip'].merge(data1['Household'][['hhno', 'hhtaz']], on='hhno')
    tp_hh_2 = data2['Trip_cloned'].merge(data2['Household'][['hhno', 'hhtaz']], on='hhno')

    tour_ok_1 = tu_hh_1.\
        query('tautodist>0 and tautodist<200')[['hhno', 'pno', 'tour', 'day', 'tautodist', 'toexpfac', 'pdpurp', 'tmodetp', 'tdtaz']].copy(deep=True)
    tour_ok_2 = tu_hh_2.\
        query('tautodist>0 and tautodist<200')[['hhno', 'pno', 'tour', 'day', 'tautodist', 'toexpfac', 'pdpurp', 'tmodetp', 'tdtaz']].copy(deep=True) 
    trip_ok_1 = tp_hh_1.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'tour', 'day', 'travdist', 'trexpfac', 'dpurp', 'mode', 'dtaz']].copy(deep=True)
    trip_ok_2 = tp_hh_2.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'tour', 'day', 'travdist', 'trexpfac', 'dpurp', 'mode', 'dtaz']] .copy(deep=True)
    
    #Merge tour and trip files
    tourtrip1 = pd.merge(tour_ok_1[['hhno', 'pno', 'tour', 'day', 'tautodist', 'toexpfac', 'pdpurp', 'tmodetp']],
                       trip_ok_1[['hhno', 'pno', 'tour', 'day', 'trexpfac']],
                       on = ['hhno', 'pno', 'tour', 'day'])
    tourtrip2 = pd.merge(tour_ok_2[['hhno', 'pno', 'tour', 'day', 'tautodist', 'toexpfac', 'pdpurp', 'tmodetp']],
                       trip_ok_2[['hhno', 'pno', 'tour', 'day', 'trexpfac']],
                       on = ['hhno', 'pno', 'tour', 'day'])

    #Compute weighted average of trip length grouped by purpose
    triptotal1 = weighted_average(tourtrip1[['tautodist', 'toexpfac', 'tmodetp']], 'tautodist', 'toexpfac', 'tmodetp')
    triptotal2 = weighted_average(tourtrip2[['tautodist', 'toexpfac', 'tmodetp']], 'tautodist', 'toexpfac', 'tmodetp')

    #Create data frame
    atl1 = pd.DataFrame.from_dict({'Average Tour Length (' + name1 + ')' : triptotal1})
    atl2 = pd.DataFrame.from_dict({'Average Tour Length (' + name2 + ')': triptotal2})
    atl = pd.merge(atl1, atl2, 'outer', left_index = True, right_index = True)
    atl = get_differences(atl, 'Average Tour Length (' + name1 + ')', 'Average Tour Length (' + name2 + ')', 2)
    atl = recode_index(atl, 'tmodetp', 'Tour Mode')  
    atl = atl.loc[mode_cat.values()]
    # display table
    display(atl.style.format({
        f'Average Tour Length ({name1})': '{:,.1f}',
        f'Average Tour Length ({name2})': '{:,.1f}',
        f'Difference (Average Tour Length ({name1}) - Average Tour Length ({name2}))': '{:,.1f}',
        f'% Difference (Average Tour Length ({name1}) - Average Tour Length ({name2}))': '{:,.1f}%'
    }))
    # bar plot
    fig = px.bar(
        atl.reset_index(),
        x='Tour Mode',
        y=[f'Average Tour Length ({name1})', f'Average Tour Length ({name2})'],
        barmode='group',
        title=f'Average Tour Distance by Mode ({tag})'
    )
    fig.update_layout(yaxis_title='Average Tour Distance', xaxis_title='Tour Mode', xaxis=dict(showgrid=True), yaxis=dict(showgrid=True))
    fig.show()

In [28]:
tours_distance_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region')

,Average Tour Length (DaysimOutputs),Average Tour Length (2023Survey),Difference (Average Tour Length (DaysimOutputs) - Average Tour Length (2023Survey)),% Difference (Average Tour Length (DaysimOutputs) - Average Tour Length (2023Survey))
Tour Purpose,,,,
Work,11.6,11.8,-0.2,-2.0%
School,4.8,4.6,0.2,5.3%
Escort,7.4,7.1,0.3,4.7%
Personal Business,6.7,8.0,-1.2,-15.7%
Shop,4.8,4.7,0.1,2.0%
Meal,3.6,3.8,-0.1,-3.8%
Social,5.8,6.9,-1.1,-16.3%


In [29]:
tours_distance_by_purp(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

,Average Tour Length (DaysimOutputs),Average Tour Length (2023Survey),Difference (Average Tour Length (DaysimOutputs) - Average Tour Length (2023Survey)),% Difference (Average Tour Length (DaysimOutputs) - Average Tour Length (2023Survey))
Tour Purpose,,,,
Work,9.2,7.0,2.2,31.1%
School,3.9,4.3,-0.3,-8.2%
Escort,6.6,11.2,-4.6,-41.3%
Personal Business,7.1,5.7,1.4,25.1%
Shop,2.7,4.0,-1.3,-32.0%
Meal,2.9,5.3,-2.4,-46.0%
Social,4.9,8.0,-3.1,-39.0%


## Tour Travel Time by Purpose

In [30]:
def tours_tt_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Average distance by tour purpose
    name1 = 'DaysimOutputs'
    name2 = f'{survey_year}Survey' 
    ##Filter out unreasonable trip/tour lengths
    ##survey data does not include drive to transit trips, remove- how can we do this without referencing max internal zone?    
    tu_hh_1 = data1['Tour'].merge(data1['Household'][['hhno', 'hhtaz']], on='hhno')
    tu_hh_2 = data2['Tour_cloned'].merge(data2['Household'][['hhno', 'hhtaz']], on='hhno')
    tp_hh_1 = data1['Trip'].merge(data1['Household'][['hhno', 'hhtaz']], on='hhno')
    tp_hh_2 = data2['Trip_cloned'].merge(data2['Household'][['hhno', 'hhtaz']], on='hhno')

    tour_ok_1 = tu_hh_1.\
        query('tautodist>0 and tautodist<200')[['hhno', 'pno', 'tour', 'day', 'tautotime', 'toexpfac', 'pdpurp', 'tmodetp', 'tdtaz']].copy(deep=True)
    tour_ok_2 = tu_hh_2.\
        query('tautodist>0 and tautodist<200')[['hhno', 'pno', 'tour', 'day', 'tautotime', 'toexpfac', 'pdpurp', 'tmodetp', 'tdtaz']].copy(deep=True) 
    trip_ok_1 = tp_hh_1.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'tour', 'day', 'travtime', 'trexpfac', 'dpurp', 'mode', 'dtaz']].copy(deep=True)
    trip_ok_2 = tp_hh_2.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'tour', 'day', 'travtime', 'trexpfac', 'dpurp', 'mode', 'dtaz']] .copy(deep=True)
    
    #Merge tour and trip files
    tourtrip1 = pd.merge(tour_ok_1[['hhno', 'pno', 'tour', 'day', 'tautotime', 'toexpfac', 'pdpurp', 'tmodetp']],
                       trip_ok_1[['hhno', 'pno', 'tour', 'day', 'trexpfac']],
                       on = ['hhno', 'pno', 'tour', 'day'])
    tourtrip2 = pd.merge(tour_ok_2[['hhno', 'pno', 'tour', 'day', 'tautotime', 'toexpfac', 'pdpurp', 'tmodetp']],
                       trip_ok_2[['hhno', 'pno', 'tour', 'day', 'trexpfac']],
                       on = ['hhno', 'pno', 'tour', 'day'])

    #Compute weighted average of trip length grouped by purpose
    triptotal1 = weighted_average(tourtrip1[['tautotime', 'toexpfac', 'pdpurp']], 'tautotime', 'toexpfac', 'pdpurp')
    triptotal2 = weighted_average(tourtrip2[['tautotime', 'toexpfac', 'pdpurp']], 'tautotime', 'toexpfac', 'pdpurp')

    #Create data frame
    atl1 = pd.DataFrame.from_dict({'Average Tour Travel Time (' + name1 + ')' : triptotal1})
    atl2 = pd.DataFrame.from_dict({'Average Tour Travel Time (' + name2 + ')': triptotal2})
    atl = pd.merge(atl1, atl2, 'outer', left_index = True, right_index = True)
    atl = get_differences(atl, 'Average Tour Travel Time (' + name1 + ')', 'Average Tour Travel Time (' + name2 + ')', 2)
    atl = recode_index(atl, 'pdpurp', 'Tour Purpose')  
    atl.columns.name = 'Travel Time (minutes)'
    atl = atl.loc[pdpurp_cat.values(), :]
    # display table
    display(atl.style.format({
        f'Average Tour Travel Time ({name1})': '{:,.1f}',
        f'Average Tour Travel Time ({name2})': '{:,.1f}',
        f'Difference (Average Tour Travel Time ({name1}) - Average Tour Travel Time ({name2}))': '{:,.1f}',
        f'% Difference (Average Tour Travel Time ({name1}) - Average Tour Travel Time ({name2}))': '{:,.1f}%'
    }))
    # bar plot
    fig = px.bar(
        atl.reset_index(),
        x='Tour Purpose',
        y=[f'Average Tour Travel Time ({name1})', f'Average Tour Travel Time ({name2})'],
        barmode='group',
        title=f'Average Tour Travel Time by Purpose ({tag})'
    )
    fig.update_layout(yaxis_title='Average Tour Travel Time', xaxis_title='Tour Purpose', xaxis=dict(showgrid=True), yaxis=dict(showgrid=True))
    fig.show()

In [31]:
tours_tt_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region')

Travel Time (minutes),Average Tour Travel Time (DaysimOutputs),Average Tour Travel Time (2023Survey),Difference (Average Tour Travel Time (DaysimOutputs) - Average Tour Travel Time (2023Survey)),% Difference (Average Tour Travel Time (DaysimOutputs) - Average Tour Travel Time (2023Survey))
Tour Purpose,,,,
Work,29.0,25.3,3.7,14.5%
School,15.2,13.4,1.8,13.5%
Escort,21.2,16.1,5.1,31.6%
Personal Business,20.1,37.6,-17.5,-46.5%
Shop,16.5,14.4,2.1,14.8%
Meal,13.6,16.7,-3.1,-18.8%
Social,17.6,17.4,0.1,0.7%


In [32]:
tours_tt_by_purp(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

Travel Time (minutes),Average Tour Travel Time (DaysimOutputs),Average Tour Travel Time (2023Survey),Difference (Average Tour Travel Time (DaysimOutputs) - Average Tour Travel Time (2023Survey)),% Difference (Average Tour Travel Time (DaysimOutputs) - Average Tour Travel Time (2023Survey))
Tour Purpose,,,,
Work,18.9,17.5,1.5,8.4%
School,8.8,12.1,-3.2,-26.8%
Escort,13.8,23.2,-9.4,-40.7%
Personal Business,15.0,14.1,0.9,6.5%
Shop,6.8,12.2,-5.4,-44.0%
Meal,7.2,13.5,-6.4,-47.0%
Social,10.8,20.6,-9.9,-48.0%


## Tour Travel Time by Mode

In [33]:
def tours_tt_by_mode(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Average distance by tour purpose
    name1 = 'DaysimOutputs'
    name2 = f'{survey_year}Survey' 
    ##Filter out unreasonable trip/tour lengths
    ##survey data does not include drive to transit trips, remove- how can we do this without referencing max internal zone?    
    tu_hh_1 = data1['Tour'].merge(data1['Household'][['hhno', 'hhtaz']], on='hhno')
    tu_hh_2 = data2['Tour_cloned'].merge(data2['Household'][['hhno', 'hhtaz']], on='hhno')
    tp_hh_1 = data1['Trip'].merge(data1['Household'][['hhno', 'hhtaz']], on='hhno')
    tp_hh_2 = data2['Trip_cloned'].merge(data2['Household'][['hhno', 'hhtaz']], on='hhno')

    tour_ok_1 = tu_hh_1.\
        query('tautodist>0 and tautodist<200')[['hhno', 'pno', 'tour', 'day', 'tautotime', 'toexpfac', 'pdpurp', 'tmodetp', 'tdtaz']].copy(deep=True)
    tour_ok_2 = tu_hh_2.\
        query('tautodist>0 and tautodist<200')[['hhno', 'pno', 'tour', 'day', 'tautotime', 'toexpfac', 'pdpurp', 'tmodetp', 'tdtaz']].copy(deep=True) 
    trip_ok_1 = tp_hh_1.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'tour', 'day', 'travtime', 'trexpfac', 'dpurp', 'mode', 'dtaz']].copy(deep=True)
    trip_ok_2 = tp_hh_2.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'tour', 'day', 'travtime', 'trexpfac', 'dpurp', 'mode', 'dtaz']] .copy(deep=True)
    
    #Merge tour and trip files
    tourtrip1 = pd.merge(tour_ok_1[['hhno', 'pno', 'tour', 'day', 'tautotime', 'toexpfac', 'tmodetp']],
                       trip_ok_1[['hhno', 'pno', 'tour', 'day', 'trexpfac']],
                       on = ['hhno', 'pno', 'tour', 'day'])
    tourtrip2 = pd.merge(tour_ok_2[['hhno', 'pno', 'tour', 'day', 'tautotime', 'toexpfac', 'tmodetp']],
                       trip_ok_2[['hhno', 'pno', 'tour', 'day', 'trexpfac']],
                       on = ['hhno', 'pno', 'tour', 'day'])

    #Compute weighted average of trip length grouped by purpose
    triptotal1 = weighted_average(tourtrip1[['tautotime', 'toexpfac', 'tmodetp']], 'tautotime', 'toexpfac', 'tmodetp')
    triptotal2 = weighted_average(tourtrip2[['tautotime', 'toexpfac', 'tmodetp']], 'tautotime', 'toexpfac', 'tmodetp')

    #Create data frame
    atl1 = pd.DataFrame.from_dict({'Average Tour Travel Time (' + name1 + ')' : triptotal1})
    atl2 = pd.DataFrame.from_dict({'Average Tour Travel Time (' + name2 + ')': triptotal2})
    atl = pd.merge(atl1, atl2, 'outer', left_index = True, right_index = True)
    atl = get_differences(atl, 'Average Tour Travel Time (' + name1 + ')', 'Average Tour Travel Time (' + name2 + ')', 2)
    atl = recode_index(atl, 'tmodetp', 'Tour Mode')  
    atl.columns.name = 'Travel Time (minutes)'
    atl = atl.loc[mode_cat.values(), :]
    # display table
    display(atl.style.format({
        f'Average Tour Travel Time ({name1})': '{:,.1f}',
        f'Average Tour Travel Time ({name2})': '{:,.1f}',
        f'Difference (Average Tour Travel Time ({name1}) - Average Tour Travel Time ({name2}))': '{:,.1f}',
        f'% Difference (Average Tour Travel Time ({name1}) - Average Tour Travel Time ({name2}))': '{:,.1f}%'
    }))
    
    # bar plot
    fig = px.bar(
        atl.reset_index(),
        x='Tour Mode',
        y=[f'Average Tour Travel Time ({name1})', f'Average Tour Travel Time ({name2})'],
        barmode='group',
        title=f'Average Tour Travel Time by Mode ({tag})'
    )
    fig.update_layout(yaxis_title='Average Tour Travel Time', xaxis_title='Tour Mode', xaxis=dict(showgrid=True), yaxis=dict(showgrid=True))
    fig.show()

In [34]:
tours_tt_by_mode(data1=data_daysim, data2=data_survey, tag='PSRC Region')

Travel Time (minutes),Average Tour Travel Time (DaysimOutputs),Average Tour Travel Time (2023Survey),Difference (Average Tour Travel Time (DaysimOutputs) - Average Tour Travel Time (2023Survey)),% Difference (Average Tour Travel Time (DaysimOutputs) - Average Tour Travel Time (2023Survey))
Tour Mode,,,,
Walk,3.6,19.1,-15.4,-81.0%
Bike,9.0,11.9,-2.9,-24.1%
SOV,24.5,19.6,5.0,25.4%
HOV2,22.2,17.7,4.5,25.4%
HOV3+,22.4,17.8,4.6,25.8%
Transit Walk Access,27.1,67.9,-40.8,-60.1%
Transit Auto Access,36.7,37.9,-1.3,-3.4%
School Bus,12.9,10.5,2.4,22.8%


In [35]:
tours_tt_by_mode(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

Travel Time (minutes),Average Tour Travel Time (DaysimOutputs),Average Tour Travel Time (2023Survey),Difference (Average Tour Travel Time (DaysimOutputs) - Average Tour Travel Time (2023Survey)),% Difference (Average Tour Travel Time (DaysimOutputs) - Average Tour Travel Time (2023Survey))
Tour Mode,,,,
Walk,2.7,12.7,-9.9,-78.6%
Bike,7.5,10.2,-2.7,-26.7%
SOV,14.4,14.3,0.1,0.8%
HOV2,13.5,23.7,-10.2,-43.1%
HOV3+,13.2,16.4,-3.2,-19.4%
Transit Walk Access,13.8,22.7,-8.9,-39.2%
Transit Auto Access,15.6,37.5,-21.9,-58.3%
School Bus,6.9,8.7,-1.8,-20.3%


## Tours by District

In [36]:
##Tours per Person by Purpose and Person Type/Number of Stops
data1=data_daysim
data2=data_survey
name1 = 'DaysimOutputs'
name2 = f'{survey_year}Survey' 
# calculate the percentage of each number of stops by purpose
data1['Household'] = data1['Household'].merge(taz_subarea[['TAZ', 'DistrictFlowName']], left_on='hhtaz', right_on='TAZ', how='left')
data1['Tour'] = data1['Tour'].merge(data1['Household'][['hhno', 'DistrictFlowName', 'hhincome', 'hhvehs']], on='hhno', how='left')
data2['Household'] = data2['Household'].merge(taz_subarea[['TAZ', 'DistrictFlowName']], left_on='hhtaz', right_on='TAZ', how='left')
data2['Tour_cloned'] = data2['Tour_cloned'].merge(data2['Household'][['hhno', 'DistrictFlowName', 'hhincome', 'hhvehs']], on='hhno', how='left')
todist1 = data1['Tour'].groupby(by='DistrictFlowName')['toexpfac'].sum()
todist2 = data2['Tour_cloned'].groupby(by='DistrictFlowName')['toexpfac'].sum()
df_compare = pd.concat([todist1, todist2], axis=1)
df_compare.columns = [name1, name2]
df_compare['Difference'] = df_compare[name1] - df_compare[name2]
df_compare['% Difference'] = (df_compare['Difference'] / df_compare[name2]) * 100
display(df_compare.loc[district_flow_name.values()].style.format({
    name1: '{:,.0f}',
    name2: '{:,.0f}',
    'Difference': '{:,.0f}',
    '% Difference': '{:,.1f}%'
}))

,DaysimOutputs,2023Survey,Difference,% Difference
DistrictFlowName,,,,
Bellevue (excluding downtown),"195,227","156,198","39,029",25.0%
Bellevue Downtown,"27,202","7,684","19,518",254.0%
Kirkland,"134,270","61,197","73,073",119.4%
Redmond,"105,474","107,608","-2,134",-2.0%
Seattle (excluding Seattle downtown),"1,022,581","799,965","222,616",27.8%
Seattle downtown,"149,809","73,367","76,442",104.2%
Rest,"4,240,924","3,003,645","1,237,279",41.2%


In [37]:
def tours_by_gp1_by_gp2(data1=data_daysim, data2=data_survey, 
                        gp1='DistrictFlowName', gp2='pdpurp', 
                        gp1_label='DistrictFlowName', gp2_label='Tour Purpose',
                        gp1_list=[], gp2_list=[]):
    tour_by_district_purpose1 = data1['Tour'].groupby([gp1, gp2], observed=False)['toexpfac'].sum().unstack(gp2)
    tour_by_district_purpose2 = data2['Tour_cloned'].groupby([gp1, gp2], observed=False)['toexpfac'].sum().unstack(gp2)
    # Reorder index and columns as requested
    tour_by_district_purpose1 = tour_by_district_purpose1.loc[list(gp1_list), list(gp2_list)]
    tour_by_district_purpose2 = tour_by_district_purpose2.loc[list(gp1_list), list(gp2_list)]
    tour_by_district_purpose1.columns.name = gp2_label
    tour_by_district_purpose2.columns.name = gp2_label
    tour_by_district_purpose1.index.name = gp1_label
    tour_by_district_purpose2.index.name = gp1_label
    display(tour_by_district_purpose1.style.format('{:,.0f}').set_caption("DaysimOutputs"))
    display(tour_by_district_purpose2.style.format('{:,.0f}').set_caption(f"{survey_year}Survey"))
    percent_diff = (tour_by_district_purpose1 - tour_by_district_purpose2) / tour_by_district_purpose2 * 100
    display(percent_diff.style.format('{:,.1f}%').set_caption("Percentage Difference (DaysimOutputs - Survey)"))

In [38]:
def tour_share_by_gp1_by_gp2(data1=data_daysim, data2=data_survey, 
                        gp1='DistrictFlowName', gp2='pdpurp', 
                        gp1_label='DistrictFlowName', gp2_label='Tour Purpose',
                        gp1_list=[], gp2_list=[]):
    tour_by_district_purpose1 = data1['Tour'].groupby([gp1, gp2], observed=False)['toexpfac'].sum().unstack(gp2)
    tour_by_district_purpose2 = data2['Tour_cloned'].groupby([gp1, gp2], observed=False)['toexpfac'].sum().unstack(gp2)
    # Reorder index and columns as requested
    tour_by_district_purpose1 = tour_by_district_purpose1.loc[list(gp1_list), list(gp2_list)]
    tour_by_district_purpose2 = tour_by_district_purpose2.loc[list(gp1_list), list(gp2_list)]
    tour_by_district_purpose1.columns.name = gp2_label
    tour_by_district_purpose2.columns.name = gp2_label
    tour_by_district_purpose1.index.name = gp1_label
    tour_by_district_purpose2.index.name = gp1_label
    tour_by_district_purpose1 = tour_by_district_purpose1.div(tour_by_district_purpose1.sum(axis=0), axis=1) * 100
    tour_by_district_purpose2 = tour_by_district_purpose2.div(tour_by_district_purpose2.sum(axis=0), axis=1) * 100
    display(tour_by_district_purpose1.style.format('{:,.1f}%').set_caption("DaysimOutputs"))
    display(tour_by_district_purpose2.style.format('{:,.1f}%').set_caption(f"{survey_year}Survey"))

## Tours by District by Purpose

In [39]:
tours_by_gp1_by_gp2(data1=data_daysim, data2=data_survey, 
                        gp1='pdpurp', gp2='DistrictFlowName', 
                        gp1_label='Tour Purpose', gp2_label='DistrictFlowName',
                        gp1_list=pdpurp_cat.values(), gp2_list=district_flow_name.values())

DistrictFlowName,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest
Tour Purpose,,,,,,,
Work,"45,525","6,213","35,198","27,689","323,790","57,678","1,122,333"
School,"24,541","1,781","13,661","10,752","99,696","5,684","416,639"
Escort,"21,844","2,336","14,260","11,288","85,481","7,314","506,908"
Personal Business,"12,361","2,384","8,826","6,899","65,691","10,262","305,826"
Shop,"19,364","3,258","13,308","10,458","97,179","14,797","600,617"
Meal,"16,568","3,033","11,613","9,105","86,371","15,218","275,246"
Social,"55,024","8,197","37,404","29,283","264,373","38,856","1,013,355"


DistrictFlowName,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest
Tour Purpose,,,,,,,
Work,"42,306","1,628","21,347","26,193","216,041","18,412","840,701"
School,"15,741",460,"6,663","17,917","89,718","2,193","326,254"
Escort,"19,935",176,"11,294","6,841","100,471","5,755","319,216"
Personal Business,"8,917",825,"1,098","3,616","46,592","7,087","232,686"
Shop,"13,154","1,140","6,288","5,071","80,216","11,157","405,194"
Meal,"8,593","1,283","2,940","9,653","71,605","8,958","160,869"
Social,"47,552","2,172","11,567","38,317","195,321","19,806","718,726"


DistrictFlowName,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest
Tour Purpose,,,,,,,
Work,7.6%,281.6%,64.9%,5.7%,49.9%,213.3%,33.5%
School,55.9%,287.4%,105.0%,-40.0%,11.1%,159.1%,27.7%
Escort,9.6%,"1,230.1%",26.3%,65.0%,-14.9%,27.1%,58.8%
Personal Business,38.6%,189.0%,703.8%,90.8%,41.0%,44.8%,31.4%
Shop,47.2%,185.7%,111.6%,106.2%,21.1%,32.6%,48.2%
Meal,92.8%,136.3%,295.0%,-5.7%,20.6%,69.9%,71.1%
Social,15.7%,277.3%,223.4%,-23.6%,35.4%,96.2%,41.0%


In [40]:
tour_share_by_gp1_by_gp2(data1=data_daysim, data2=data_survey, 
                        gp1='pdpurp', gp2='DistrictFlowName', 
                        gp1_label='Tour Purpose', gp2_label='DistrictFlowName',
                        gp1_list=pdpurp_cat.values(), gp2_list=district_flow_name.values())

DistrictFlowName,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest
Tour Purpose,,,,,,,
Work,23.3%,22.8%,26.2%,26.3%,31.7%,38.5%,26.5%
School,12.6%,6.5%,10.2%,10.2%,9.7%,3.8%,9.8%
Escort,11.2%,8.6%,10.6%,10.7%,8.4%,4.9%,12.0%
Personal Business,6.3%,8.8%,6.6%,6.5%,6.4%,6.9%,7.2%
Shop,9.9%,12.0%,9.9%,9.9%,9.5%,9.9%,14.2%
Meal,8.5%,11.1%,8.6%,8.6%,8.4%,10.2%,6.5%
Social,28.2%,30.1%,27.9%,27.8%,25.9%,25.9%,23.9%


DistrictFlowName,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest
Tour Purpose,,,,,,,
Work,27.1%,21.2%,34.9%,24.3%,27.0%,25.1%,28.0%
School,10.1%,6.0%,10.9%,16.7%,11.2%,3.0%,10.9%
Escort,12.8%,2.3%,18.5%,6.4%,12.6%,7.8%,10.6%
Personal Business,5.7%,10.7%,1.8%,3.4%,5.8%,9.7%,7.7%
Shop,8.4%,14.8%,10.3%,4.7%,10.0%,15.2%,13.5%
Meal,5.5%,16.7%,4.8%,9.0%,9.0%,12.2%,5.4%
Social,30.4%,28.3%,18.9%,35.6%,24.4%,27.0%,23.9%


## Tours by District by Mode

In [41]:
tours_by_gp1_by_gp2(data1=data_daysim, data2=data_survey, 
                    gp1='DistrictFlowName', gp2='tmodetp', 
                    gp1_label='DistrictFlowName', gp2_label='Tour Mode',
                    gp1_list=district_flow_name.values(), gp2_list=mode_cat.values())

Tour Mode,Walk,Bike,SOV,HOV2,HOV3+,Transit Walk Access,Transit Auto Access,School Bus
DistrictFlowName,,,,,,,,
Bellevue (excluding downtown),"10,309",662,"72,581","47,642","42,574","14,174",673,"6,612"
Bellevue Downtown,"8,862",334,"8,153","4,608","2,041","2,774",78,352
Kirkland,"9,422",454,"53,126","33,336","26,364","8,029",261,"3,278"
Redmond,"7,303",516,"42,283","25,051","20,966","6,453",277,"2,625"
Seattle (excluding Seattle downtown),"189,263","7,974","387,732","188,050","151,237","75,688","2,184","20,453"
Seattle downtown,"83,885","1,797","34,247","12,465","5,781","11,062",239,333
Rest,"323,873","21,079","1,670,284","1,002,151","1,022,834","84,679","7,143","108,881"


Tour Mode,Walk,Bike,SOV,HOV2,HOV3+,Transit Walk Access,Transit Auto Access,School Bus
DistrictFlowName,,,,,,,,
Bellevue (excluding downtown),"24,859","1,722","40,509","52,129","28,415","4,405",nan,"1,353"
Bellevue Downtown,"1,909",187,"2,728","2,175",208,425,nan,nan
Kirkland,"9,449",nan,"25,981","10,864","8,783","3,146",421,"1,500"
Redmond,"6,735","1,627","40,100","15,781","15,944","8,441","8,364","7,973"
Seattle (excluding Seattle downtown),"143,556","29,563","252,773","206,703","103,882","50,194",385,"6,909"
Seattle downtown,"30,339","2,951","14,023","12,858","3,386","7,078",985,27
Rest,"191,968","14,026","1,157,926","673,840","720,968","115,053","4,303","88,544"


Tour Mode,Walk,Bike,SOV,HOV2,HOV3+,Transit Walk Access,Transit Auto Access,School Bus
DistrictFlowName,,,,,,,,
Bellevue (excluding downtown),-58.5%,-61.6%,79.2%,-8.6%,49.8%,221.8%,nan%,388.5%
Bellevue Downtown,364.3%,78.5%,198.8%,111.9%,881.2%,552.5%,nan%,nan%
Kirkland,-0.3%,nan%,104.5%,206.8%,200.2%,155.2%,-37.9%,118.6%
Redmond,8.4%,-68.3%,5.4%,58.7%,31.5%,-23.5%,-96.7%,-67.1%
Seattle (excluding Seattle downtown),31.8%,-73.0%,53.4%,-9.0%,45.6%,50.8%,466.6%,196.0%
Seattle downtown,176.5%,-39.1%,144.2%,-3.1%,70.7%,56.3%,-75.7%,"1,111.3%"
Rest,68.7%,50.3%,44.2%,48.7%,41.9%,-26.4%,66.0%,23.0%


In [42]:
tour_share_by_gp1_by_gp2(data1=data_daysim, data2=data_survey, 
                    gp1='DistrictFlowName', gp2='tmodetp', 
                    gp1_label='DistrictFlowName', gp2_label='Tour Mode',
                    gp1_list=district_flow_name.values(), gp2_list=mode_cat.values())

Tour Mode,Walk,Bike,SOV,HOV2,HOV3+,Transit Walk Access,Transit Auto Access,School Bus
DistrictFlowName,,,,,,,,
Bellevue (excluding downtown),1.6%,2.0%,3.2%,3.6%,3.3%,7.0%,6.2%,4.6%
Bellevue Downtown,1.4%,1.0%,0.4%,0.4%,0.2%,1.4%,0.7%,0.2%
Kirkland,1.5%,1.4%,2.3%,2.5%,2.1%,4.0%,2.4%,2.3%
Redmond,1.2%,1.6%,1.9%,1.9%,1.6%,3.2%,2.6%,1.8%
Seattle (excluding Seattle downtown),29.9%,24.3%,17.1%,14.3%,11.9%,37.3%,20.1%,14.3%
Seattle downtown,13.3%,5.5%,1.5%,0.9%,0.5%,5.5%,2.2%,0.2%
Rest,51.2%,64.2%,73.6%,76.3%,80.4%,41.7%,65.8%,76.4%


Tour Mode,Walk,Bike,SOV,HOV2,HOV3+,Transit Walk Access,Transit Auto Access,School Bus
DistrictFlowName,,,,,,,,
Bellevue (excluding downtown),6.1%,3.4%,2.6%,5.4%,3.2%,2.3%,nan%,1.3%
Bellevue Downtown,0.5%,0.4%,0.2%,0.2%,0.0%,0.2%,nan%,nan%
Kirkland,2.3%,nan%,1.7%,1.1%,1.0%,1.7%,2.9%,1.4%
Redmond,1.6%,3.2%,2.6%,1.6%,1.8%,4.5%,57.9%,7.5%
Seattle (excluding Seattle downtown),35.1%,59.0%,16.5%,21.2%,11.8%,26.6%,2.7%,6.5%
Seattle downtown,7.4%,5.9%,0.9%,1.3%,0.4%,3.8%,6.8%,0.0%
Rest,47.0%,28.0%,75.5%,69.2%,81.8%,61.0%,29.8%,83.3%


## Tours by Purpose by Auto Sufficiency

In [43]:
def classify_auto_sufficiency(row):
    if row['hhvehs'] == 0:
        return '0 veh'
    elif row['hhvehs'] < row['driver']:
        return 'autos < drivers'
    elif row['hhvehs'] == row['driver']:
        return 'autos = drivers'
    else:
        return 'autos > drivers'

data1['Trip']['driver'] = (data1['Trip']['dorp'] == 'Driver').astype(int)
data2['Trip_cloned']['driver'] = (data2['Trip_cloned']['dorp'] == 'Driver').astype(int)

_driver1 = data1['Trip'][['hhno', 'pno', 'driver']].groupby(by=['hhno', 'pno'])['driver'].sum().reset_index()
_driver1['driver'] = (_driver1['driver'] > 0).astype(int)
driver1 = _driver1.groupby(by='hhno')['driver'].sum()

_driver2 = data2['Trip_cloned'][['hhno', 'pno', 'driver']].groupby(by=['hhno', 'pno'])['driver'].sum().reset_index()
_driver2['driver'] = (_driver2['driver'] > 0).astype(int)
driver2 = _driver2.groupby(by='hhno')['driver'].sum()

data1['Tour'] = data1['Tour'].merge(driver1, on='hhno', how='left')
data2['Tour_cloned'] = data2['Tour_cloned'].merge(driver2, on='hhno', how='left')

data1['Tour']['auto_sufficiency'] = data1['Tour'][['hhvehs', 'driver']].apply(classify_auto_sufficiency, axis=1)
data2['Tour_cloned']['auto_sufficiency'] = data2['Tour_cloned'][['hhvehs', 'driver']].apply(classify_auto_sufficiency, axis=1)

In [44]:
autosuf_labels = ['0 veh', 'autos < drivers', 'autos = drivers', 'autos > drivers']

tours_by_gp1_by_gp2(data1=data_daysim, data2=data_survey, 
                    gp1='pdpurp', gp2='auto_sufficiency', 
                    gp1_label='Tour Purpose', gp2_label='Auto Sufficiency',
                    gp1_list=pdpurp_cat.values(), gp2_list=autosuf_labels)

Auto Sufficiency,0 veh,autos < drivers,autos = drivers,autos > drivers
Tour Purpose,,,,
Work,"33,162","284,772","836,102","464,390"
School,"22,731","131,997","255,681","162,345"
Escort,"24,179","140,686","309,494","175,072"
Personal Business,"37,752","65,591","206,413","102,493"
Shop,"70,211","127,374","356,400","204,996"
Meal,"19,154","83,020","203,083","111,897"
Social,"59,984","239,080","720,587","426,841"


Auto Sufficiency,0 veh,autos < drivers,autos = drivers,autos > drivers
Tour Purpose,,,,
Work,"26,609","32,601","515,906","814,078"
School,"9,931","11,824","174,176","293,231"
Escort,"1,508","7,490","204,598","331,320"
Personal Business,"30,306","10,406","129,196","176,184"
Shop,"78,908","16,338","170,704","386,900"
Meal,"10,058","16,156","87,616","237,257"
Social,"20,065","19,862","342,370","895,123"


Auto Sufficiency,0 veh,autos < drivers,autos = drivers,autos > drivers
Tour Purpose,,,,
Work,24.6%,773.5%,62.1%,-43.0%
School,128.9%,"1,016.3%",46.8%,-44.6%
Escort,"1,503.3%","1,778.3%",51.3%,-47.2%
Personal Business,24.6%,530.3%,59.8%,-41.8%
Shop,-11.0%,679.6%,108.8%,-47.0%
Meal,90.4%,413.9%,131.8%,-52.8%
Social,198.9%,"1,103.7%",110.5%,-52.3%


In [45]:
tour_share_by_gp1_by_gp2(data1=data_daysim, data2=data_survey, 
                    gp1='pdpurp', gp2='auto_sufficiency', 
                    gp1_label='Tour Purpose', gp2_label='Auto Sufficiency',
                    gp1_list=pdpurp_cat.values(), gp2_list=autosuf_labels)

Auto Sufficiency,0 veh,autos < drivers,autos = drivers,autos > drivers
Tour Purpose,,,,
Work,12.4%,26.6%,29.0%,28.2%
School,8.5%,12.3%,8.9%,9.9%
Escort,9.0%,13.1%,10.7%,10.6%
Personal Business,14.1%,6.1%,7.1%,6.2%
Shop,26.3%,11.9%,12.3%,12.4%
Meal,7.2%,7.7%,7.0%,6.8%
Social,22.5%,22.3%,25.0%,25.9%


Auto Sufficiency,0 veh,autos < drivers,autos = drivers,autos > drivers
Tour Purpose,,,,
Work,15.0%,28.4%,31.8%,26.0%
School,5.6%,10.3%,10.7%,9.4%
Escort,0.9%,6.5%,12.6%,10.6%
Personal Business,17.1%,9.1%,8.0%,5.6%
Shop,44.5%,14.2%,10.5%,12.3%
Meal,5.7%,14.1%,5.4%,7.6%
Social,11.3%,17.3%,21.1%,28.6%


## Tours by Purpose by Income Level

In [46]:
income_bins = [0, 25000, 45000, 75000, float('inf')]
income_labels = ['Less than 25,000', '$25,000-$44,999', '$45,000-74,999', 'More than $75,000']
data1['Tour']['income_group'] = pd.cut(data1['Tour']['hhincome'], bins=income_bins, labels=income_labels, right=False)
data2['Tour_cloned']['income_group'] = pd.cut(data2['Tour_cloned']['hhincome'], bins=income_bins, labels=income_labels, right=False)

tours_by_gp1_by_gp2(data1=data_daysim, data2=data_survey, 
                    gp1='pdpurp', gp2='income_group', 
                    gp1_label='Tour Purpose', gp2_label='Income Level',
                    gp1_list=pdpurp_cat.values(), gp2_list=income_labels)

Income Level,"Less than 25,000","$25,000-$44,999","$45,000-74,999","More than $75,000"
Tour Purpose,,,,
Work,"93,372","183,102","332,067","1,009,567"
School,"73,818","67,380","108,083","323,260"
Escort,"75,961","77,497","134,733","361,050"
Personal Business,"86,771","73,474","81,905","169,942"
Shop,"140,029","112,976","157,947","347,744"
Meal,"40,816","54,757","84,154","237,279"
Social,"152,960","191,516","292,434","809,140"


Income Level,"Less than 25,000","$25,000-$44,999","$45,000-74,999","More than $75,000"
Tour Purpose,,,,
Work,"34,915","90,047","155,619","769,885"
School,"22,898","10,638","79,640","307,323"
Escort,"5,669","33,867","85,250","294,494"
Personal Business,"41,020","35,946","25,549","159,770"
Shop,"57,193","97,456","65,293","254,464"
Meal,"32,658","22,226","25,663","159,646"
Social,"70,500","79,700","146,545","635,093"


Income Level,"Less than 25,000","$25,000-$44,999","$45,000-74,999","More than $75,000"
Tour Purpose,,,,
Work,167.4%,103.3%,113.4%,31.1%
School,222.4%,533.4%,35.7%,5.2%
Escort,"1,240.0%",128.8%,58.0%,22.6%
Personal Business,111.5%,104.4%,220.6%,6.4%
Shop,144.8%,15.9%,141.9%,36.7%
Meal,25.0%,146.4%,227.9%,48.6%
Social,117.0%,140.3%,99.6%,27.4%


In [47]:
tour_share_by_gp1_by_gp2(data1=data_daysim, data2=data_survey, 
                    gp1='pdpurp', gp2='income_group', 
                    gp1_label='Tour Purpose', gp2_label='Income Level',
                    gp1_list=pdpurp_cat.values(), gp2_list=income_labels)

Income Level,"Less than 25,000","$25,000-$44,999","$45,000-74,999","More than $75,000"
Tour Purpose,,,,
Work,14.1%,24.1%,27.9%,31.0%
School,11.1%,8.9%,9.1%,9.9%
Escort,11.4%,10.2%,11.3%,11.1%
Personal Business,13.1%,9.7%,6.9%,5.2%
Shop,21.1%,14.9%,13.3%,10.7%
Meal,6.1%,7.2%,7.1%,7.3%
Social,23.0%,25.2%,24.5%,24.8%


Income Level,"Less than 25,000","$25,000-$44,999","$45,000-74,999","More than $75,000"
Tour Purpose,,,,
Work,13.2%,24.3%,26.7%,29.8%
School,8.6%,2.9%,13.6%,11.9%
Escort,2.1%,9.2%,14.6%,11.4%
Personal Business,15.5%,9.7%,4.4%,6.2%
Shop,21.6%,26.3%,11.2%,9.9%
Meal,12.3%,6.0%,4.4%,6.2%
Social,26.6%,21.5%,25.1%,24.6%
